## Cellule RandomForestRegressor

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import joblib
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

df = pd.read_csv('/home/greg/TARDIS/cleaned_dataset.csv', sep=';')

if 'Mois' not in df.columns:
    df['Date'] = df['Date'].astype(str).str.replace('/', '-').str.replace(' ', '-')
    df['Date'] = pd.to_datetime(df['Date'], format='mixed', errors='coerce')
    df['Mois'] = df['Date'].dt.month
    df = df.dropna(subset=['Mois'])

le_dep = LabelEncoder()
le_arr = LabelEncoder()

df['Gare_depart_enc'] = le_dep.fit_transform(df['Gare de départ'])
df['Gare_arrivee_enc'] = le_arr.fit_transform(df["Gare d'arrivée"])

X = df[['Gare_depart_enc', 'Gare_arrivee_enc', 'Mois', 'Durée moyenne du trajet']]
y = df["Retard moyen de tous les trains à l'arrivée"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

modeles = {
    "Régression Linéaire": LinearRegression(),
    "Arbre de Décision": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, random_state=42)
}

meilleur_nom = ""
meilleur_score = -float('inf')
meilleur_modele = None

for nom, modele in modeles.items():
    modele.fit(X_train, y_train)
    score = modele.score(X_test, y_test)
    print(f"{nom} -> Score R² : {score:.4f}")
    
    if score > meilleur_score:
        meilleur_score = score
        meilleur_nom = nom
        meilleur_modele = modele

print(f"\n Gagnant : {meilleur_nom} avec R² = {meilleur_score:.4f}")

joblib.dump(meilleur_modele, '/home/greg/TARDIS/model.joblib')
joblib.dump(le_dep, '/home/greg/TARDIS/le_dep.joblib')
joblib.dump(le_arr, '/home/greg/TARDIS/le_arr.joblib')

Régression Linéaire -> Score R² : 0.0396
Arbre de Décision -> Score R² : -0.1477
Random Forest -> Score R² : 0.0539
Gradient Boosting -> Score R² : 0.1045

 Gagnant : Gradient Boosting avec R² = 0.1045


['/home/greg/TARDIS/le_arr.joblib']